# Baseline Edge AI Challenge 2026
Detection (CNN grid 15x20) + intent (TextCNN), dùng chung 1 thư mục data.
Mỗi phần export int8 `.tflite` + `.cc/.h` → copy vào `esp32s3_infer_script/main/models/`.


## 1. Detection


In [ ]:
# 1. load data & thong ke class
import glob
from collections import Counter

data = "egde-ai-challenge-2026-lan-2-qualifiers"
classes = open(f"{data}/classes.txt").read().split()

cnt = Counter()
for f in glob.glob(f"{data}/train/labels/*.txt"):
    for l in open(f):
        cnt[int(l.split()[0])] += 1

print("So luong moi class:", {classes[k]: v for k, v in sorted(cnt.items())})
# nx: cac class deu nhau (~5k-5.7k), data rat can bang

In [ ]:
# 2. dataloader 
import glob, os, cv2, torch
from torch.utils.data import Dataset, DataLoader

class DetData(Dataset):
    def __init__(self, imgs):
        self.imgs = imgs
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, i):
        p = self.imgs[i]
        im = cv2.imread(p)[:, :, ::-1] / 255.0 # doc bgr -> rgb, norm 0-1
        im = torch.tensor(im, dtype=torch.float32).permute(2, 0, 1)
        
        # grid 15x20: [obj, tx, ty, w, h, cls]
        # dung offset tx, ty trong cell de cnn nho hoc de hon
        tgt = torch.zeros(6, 15, 20)
        lp = p.replace("/images/", "/labels/").replace(".png", ".txt")
        if os.path.exists(lp):
            for l in open(lp):
                c, x, y, w, h = map(float, l.split())
                gx = min(int(x * 20), 19)
                gy = min(int(y * 15), 14)
                tx = x * 20 - gx # offset trong cell [0, 1]
                ty = y * 15 - gy # offset trong cell [0, 1]
                tgt[:, gy, gx] = torch.tensor([1.0, tx, ty, w, h, c])
        return im, tgt

imgs = sorted(glob.glob(f"{data}/train/images/*.png"))
train_loader = DataLoader(DetData(imgs[:5000]), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(DetData(imgs[5000:]), batch_size=32)
print("train:", len(train_loader.dataset), "val:", len(val_loader.dataset))

In [ ]:
# 3. model
import torch.nn as nn

backbone = nn.Sequential(
    nn.Conv2d(3, 16, 3, 1, 1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 24, 3, 1, 1), nn.BatchNorm2d(24), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(24, 36, 3, 1, 1), nn.BatchNorm2d(36), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(36, 36, 3, 1, 1), nn.BatchNorm2d(36), nn.ReLU(), nn.MaxPool2d(2)
)

head = nn.Sequential(
    nn.Conv2d(36, 28, 3, 1, 1), nn.BatchNorm2d(28), nn.ReLU(),
    nn.Conv2d(28, 12, 1)
)

nn.init.constant_(head[3].bias[3:5], -2.0)
nn.init.constant_(head[3].bias[0], -2.0)

model = nn.Sequential(backbone, head)
print("backbone:", sum(p.numel() for p in backbone.parameters()))
print("head    :", sum(p.numel() for p in head.parameters()))
print("total   :", sum(p.numel() for p in model.parameters()))

In [ ]:
# 4. train
from tqdm import tqdm
import torch
import torch.nn.functional as F
from torchvision.ops import box_iou, batched_nms, generalized_box_iou_loss
from torchmetrics.detection import MeanAveragePrecision

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
epochs, patience = 10,5
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
gy, gx = (grid.to(device) for grid in torch.meshgrid(torch.arange(15), torch.arange(20), indexing="ij"))

def decode_boxes(bx):
    cx = ((bx[0] if bx.dim() == 3 else bx[:, 0]) + gx) / 20
    cy = ((bx[1] if bx.dim() == 3 else bx[:, 1]) + gy) / 15
    w, h = (bx[2], bx[3]) if bx.dim() == 3 else (bx[:, 2], bx[:, 3])
    return torch.stack([(cx - w/2).clamp(0, 1), (cy - h/2).clamp(0, 1),
                        (cx + w/2).clamp(0, 1), (cy + h/2).clamp(0, 1)], dim=-1)

def evaluate(conf_th=0.3):
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox")
    metric.warn_on_many_detections = False
    ious_list = []
    tp_cnt, fp_cnt, gt_cnt = 0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            out = model(x.to(device))
            for pred, target in zip(out, y.to(device)):
                gm = target[0] == 1
                gt_b = decode_boxes(target[1:5])[gm]
                gt_c = target[5][gm].long()
                gt_cnt += len(gt_b)
                if len(gt_b):
                    ious_list.extend(box_iou(decode_boxes(pred[1:5].sigmoid())[gm], gt_b).diag().tolist())

                boxes = decode_boxes(pred[1:5].sigmoid()).reshape(-1, 4)
                prob, labels = pred[5:].softmax(dim=0).max(dim=0)
                scores = (pred[0].sigmoid() * prob).flatten()
                labels = labels.flatten()
                mask = scores > 0.05
                boxes, scores, labels = boxes[mask], scores[mask], labels[mask]
                if len(boxes):
                    keep = batched_nms(boxes, scores, labels, 0.5)
                    boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
                metric.update([{'boxes': boxes.cpu(), 'scores': scores.cpu(), 'labels': labels.cpu()}],
                              [{'boxes': gt_b.cpu(), 'labels': gt_c.cpu()}])

                mask = scores >= conf_th
                pb, ps, pc = boxes[mask], scores[mask], labels[mask]
                if not len(pb):
                    continue
                if not len(gt_b):
                    fp_cnt += len(pb)
                    continue
                ious = box_iou(pb, gt_b)
                used = torch.zeros(len(gt_b), dtype=torch.bool, device=device)
                for i in ps.argsort(descending=True):
                    row = ious[i].clone()
                    row[(gt_c != pc[i]) | used] = -1
                    best_iou, gi = row.max(dim=0)
                    if best_iou >= 0.5:
                        tp_cnt += 1
                        used[gi] = True
                    else:
                        fp_cnt += 1
    p = tp_cnt / (tp_cnt + fp_cnt + 1e-6)
    r = tp_cnt / (gt_cnt + 1e-6)
    res = metric.compute()
    print(f"TP={tp_cnt} FP={fp_cnt} GT={gt_cnt}")
    return (sum(ious_list) / (len(ious_list) + 1e-6), res['map_50'].item(),
            res['map'].item(), 2 * p * r / (p + r + 1e-6))

best_map, bad_epochs = 0.0, 0
for ep in range(epochs):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {ep+1}/{epochs}")
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        out = model(x)
        m = y[:, 0] == 1
        bce = F.binary_cross_entropy_with_logits(out[:, 0], y[:, 0], reduction="none")
        l_obj = (bce[m].sum() + 0.5 * bce[~m].sum()) / max(m.sum().item(), 1.0)
        box, target = out[:, 1:5].sigmoid(), y[:, 1:5]
        loss = (l_obj
                + 5.0 * generalized_box_iou_loss(decode_boxes(box)[m], decode_boxes(target)[m], reduction="mean")
                + 2.0 * F.l1_loss(box.permute(0, 2, 3, 1)[m], target.permute(0, 2, 3, 1)[m])
                + F.cross_entropy(out[:, 5:].permute(0, 2, 3, 1)[m], y[:, 5].long()[m]))
        opt.zero_grad()
        loss.backward()
        opt.step()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    scheduler.step()
    iou_val, map50_val, map50_95_val, f1_val = evaluate()
    print(f"[Val] IoU: {iou_val:.4f} | mAP@50: {map50_val:.4f} | mAP50-95: {map50_95_val:.4f} | F1: {f1_val:.4f}")
    if map50_val > best_map:
        best_map, bad_epochs = map50_val, 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"-> Luu best_model.pth (best mAP@50: {best_map:.4f})")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"-> Early stopping tai epoch {ep+1}!")
            break


In [ ]:
import numpy as np
import tensorflow as tf
import onnx2tf.onnx2tf as otf

model.cpu().eval().load_state_dict(torch.load("best_model.pth", map_location="cpu", weights_only=True))
torch.onnx.export(model, torch.randn(1, 3, 240, 320), "model.onnx", input_names=["input"], output_names=["output"])
otf.download_test_image_data = lambda: np.zeros((20, 128, 128, 3), dtype=np.float32)
otf.convert("model.onnx", output_folder_path="saved_model", not_use_onnxsim=True, non_verbose=True)

converter = tf.lite.TFLiteConverter.from_saved_model("saved_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: ([val_loader.dataset[i][0].permute(1, 2, 0).unsqueeze(0).numpy()] for i in range(100))
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = converter.inference_output_type = tf.int8
tflite_model = converter.convert()
open("model_int8.tflite", "wb").write(tflite_model)
print("Quantized:", len(tflite_model), "bytes")

# --- nhung vao ESP32-S3: sinh mang C, copy vao esp32s3_infer_script/main/models/ ---
cc_dir = "."
def to_cc(data, name):
    rows = ",\n".join("  " + ", ".join(f"0x{b:02x}" for b in data[i:i + 16]) for i in range(0, len(data), 16))
    open(f"{cc_dir}/{name}.h", "w").write(f"#pragma once\nextern const unsigned char {name}[];\nextern const unsigned int {name}_len;\n")
    open(f"{cc_dir}/{name}.cc", "w").write(f'#include "{name}.h"\nalignas(16) const unsigned char {name}[] = {{\n{rows}\n}};\nconst unsigned int {name}_len = {len(data)};\n')
    print(f"Saved {name}.cc/.h ({len(data)} bytes)")
to_cc(tflite_model, "det_model")


In [ ]:
import numpy as np
from ai_edge_litert.interpreter import Interpreter

interp = Interpreter(model_path="model_int8.tflite")
interp.allocate_tensors()
inp, output = interp.get_input_details()[0], interp.get_output_details()[0]
img_path = imgs[5000]
im = cv2.imread(img_path)[:, :, ::-1] / 255.0
scale, zero = inp["quantization"]
im_q = np.clip(np.round(im[None] / scale + zero), -128, 127).astype(np.int8)
interp.set_tensor(inp["index"], im_q)
interp.invoke()
scale, zero = output["quantization"]
pred = torch.from_numpy((interp.get_tensor(output["index"])[0].astype(np.float32) - zero) * scale).permute(2, 0, 1).to(device)
prob, cls_ids = pred[5:].softmax(0).max(0)
scores = (pred[0].sigmoid() * prob).flatten()
boxes, cls_ids = decode_boxes(pred[1:5].sigmoid()).reshape(-1, 4), cls_ids.flatten()
mask = scores >= 0.3
b, s, c = boxes[mask], scores[mask], cls_ids[mask]
if len(b):
    keep = batched_nms(b, s, c, 0.5)
    b, s, c = b[keep], s[keep], c[keep]
print(f"Detections: {len(b)}")
for box, score, label in zip(b, s, c):
    print(f"{classes[label]}: conf={score:.3f}, box={[round(v, 3) for v in box.tolist()]}")

## 2. Intent (TextCNN)


In [ ]:
import re, pandas as pd, torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

df = pd.read_csv(f"{data}/intent_train.csv")
torch.manual_seed(42)
order = torch.randperm(len(df)).tolist()
train, val = df.iloc[order[:4800]], df.iloc[order[4800:]]
tokens = lambda text: re.findall(r"\w+", text.lower())
vocab = {word: i + 2 for i, word in enumerate(sorted({w for q in train.question for w in tokens(q)}))}
labels = sorted(df.intent.unique().tolist())
label_ids = {name: i for i, name in enumerate(labels)}
def encode(texts):
    ids = [[vocab.get(w, 1) for w in tokens(text)] for text in texts]
    return torch.tensor([(row + [0] * 32)[:32] for row in ids], dtype=torch.long)
train_loader = DataLoader(TensorDataset(encode(train.question), torch.tensor([label_ids[y] for y in train.intent])), batch_size=64, shuffle=True)
val_x, val_y = encode(val.question), torch.tensor([label_ids[y] for y in val.intent])
print(len(vocab), len(labels), len(train), len(val))

In [ ]:
class TextCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab) + 2, 32, padding_idx=0)
        self.conv = nn.Conv1d(32, 32, 3, padding=1)
        self.fc = nn.Linear(32, len(labels))

    def forward(self, x):
        features = self.conv(self.embedding(x).transpose(1, 2)).relu()
        return self.fc(features.amax(dim=-1))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TextCNN().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
best = 0
for epoch in range(10):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = nn.functional.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        accuracy = (model(val_x.to(device)).argmax(1).cpu() == val_y).float().mean().item()
    print(f"Epoch {epoch + 1}: val accuracy {accuracy:.3f}")
    if accuracy > best:
        best = accuracy
        torch.save({"model": model.state_dict(), "vocab": vocab, "labels": labels}, "best_textcnn.pth")

In [ ]:
model.load_state_dict(torch.load("best_textcnn.pth", map_location=device, weights_only=True)["model"])
model.eval()
test = pd.read_csv(f"{data}/intent_test.csv")
with torch.no_grad():
    pred = torch.cat([model(x.to(device)).argmax(1).cpu() for x in DataLoader(encode(test.question), batch_size=256)])
test.assign(intent=[labels[i] for i in pred.tolist()]).head()

In [ ]:
import numpy as np, tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

# Dung graph TF truc tiep tu trong so PyTorch, KHONG qua onnx2tf: onnx2tf chen LESS/SELECT/CAST int64
# cho Gather (embedding) -> TFLite tren PC chay duoc nhung TFLite Micro (ESP32) khong co cac op do.
model.cpu().eval()
w = lambda p: p.detach().numpy()
E = tf.constant(w(model.embedding.weight))                        # [V+2, 32]
K = tf.constant(w(model.conv.weight).transpose(2, 1, 0))         # torch [out, in, k] -> tf [k, in, out]
cb, FW, fb = tf.constant(w(model.conv.bias)), tf.constant(w(model.fc.weight).T), tf.constant(w(model.fc.bias))

@tf.function(input_signature=[tf.TensorSpec([1, 32], tf.int32, name="tokens")])
def textcnn(tokens):
    x = tf.gather(E, tokens)                                      # [1, 32 tu, 32 emb]
    x = tf.nn.relu(tf.nn.conv1d(x, K, 1, "SAME") + cb)
    return tf.matmul(tf.reduce_max(x, axis=1), FW) + fb          # max theo chieu cau

def acc(path, n=None):
    it = Interpreter(model_path=path); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    n = n or len(val_x); ok = 0
    for k in range(n):
        it.set_tensor(i["index"], val_x[k:k+1].numpy().astype(np.int32)); it.invoke()
        ok += int(it.get_tensor(o["index"])[0].argmax()) == int(val_y[k])
    return ok / n, it

c = tf.lite.TFLiteConverter.from_concrete_functions([textcnn.get_concrete_function()], textcnn)
open("textcnn_f32_chk.tflite", "wb").write(c.convert())
print("float32 acc:", acc("textcnn_f32_chk.tflite", 500)[0])

rng = np.random.default_rng(0)
idx = rng.permutation(len(val_x))[:500]
def rep_data():
    for i in idx:
        yield [val_x[int(i):int(i)+1].numpy().astype(np.int32)]

converter = tf.lite.TFLiteConverter.from_concrete_functions([textcnn.get_concrete_function()], textcnn)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# input giu int32 (token id, vocab > 127 khong vua int8; TFLite Micro khong ho tro int64); trong so + output int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()
open("textcnn_int8.tflite", "wb").write(tflite_model)
q, interp = acc("textcnn_int8.tflite")
print("INT8 acc:", q, "|", len(tflite_model), "bytes")

# chi dung op ma firmware dang ky (main/model_config.h REGISTER_OPS) -> bat loi truoc khi flash
ops = {d["op_name"] for d in interp._get_ops_details()}
esp_ops = {"CONV_2D", "MAX_POOL_2D", "LESS", "CAST", "MUL", "ADD", "GATHER", "TRANSPOSE", "PAD", "RESHAPE",
           "REDUCE_MAX", "FULLY_CONNECTED"}
assert ops <= esp_ops, f"op ESP32 khong chay duoc: {ops - esp_ops}"
print("ops:", sorted(ops))

# --- nhung vao ESP32-S3 (to_cc, cc_dir dinh nghia o o export detection) ---
to_cc(tflite_model, "textcnn_model")

# vocab: vocab[k] co token id k+2 (0 = pad, 1 = unknown); firmware hard-code thu tu labels
assert labels == ["COUNT", "EXIST", "LIST", "LOCATION", "OUT_OF_SCOPE"], labels
words = sorted(vocab, key=vocab.get)
assert [vocab[w] for w in words] == list(range(2, len(words) + 2))
open(f"{cc_dir}/vocab.h", "w").write("#pragma once\nextern const char *const vocab[];\nextern const int vocab_len;\n")
open(f"{cc_dir}/vocab.cc", "w").write('#include "vocab.h"\nconst char *const vocab[] = {\n' + ",\n".join(f'  "{w}"' for w in words) + f"\n}};\nconst int vocab_len = {len(words)};\n")
print(f"Saved vocab.cc/.h ({len(words)} tu)")


## 3. Submission (cần cả 2 model)


In [ ]:
import csv, json, re
import numpy as np, cv2, torch
from torchvision.ops import batched_nms
from ai_edge_litert.interpreter import Interpreter

root = data
checkpoint = torch.load("best_textcnn.pth", map_location="cpu", weights_only=True)
vocab, intents = checkpoint["vocab"], checkpoint["labels"]
image_model = Interpreter(model_path="model_int8.tflite")
text_model = Interpreter(model_path="textcnn_int8.tflite")
image_model.allocate_tensors()
text_model.allocate_tensors()
image_input, image_output = image_model.get_input_details()[0], image_model.get_output_details()[0]
text_input, text_output = text_model.get_input_details()[0], text_model.get_output_details()[0]
gy, gx = torch.meshgrid(torch.arange(15), torch.arange(20), indexing="ij")

def detect(path):
    im = cv2.imread(path)
    if im is None:
        raise FileNotFoundError(path)
    h, w = im.shape[:2]
    im = cv2.cvtColor(cv2.resize(im, (320, 240)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    scale, zero = image_input["quantization"]
    image_model.set_tensor(image_input["index"], np.clip(np.rint(im[None] / scale + zero), -128, 127).astype(np.int8))
    image_model.invoke()
    scale, zero = image_output["quantization"]
    out = torch.from_numpy((image_model.get_tensor(image_output["index"])[0].astype(np.float32) - zero) * scale).permute(2, 0, 1)
    box = out[1:5].sigmoid()
    cx, cy = (box[0] + gx) / 20, (box[1] + gy) / 15
    boxes = torch.stack(((cx - box[2]/2).clamp(0, 1) * w, (cy - box[3]/2).clamp(0, 1) * h,
                         (cx + box[2]/2).clamp(0, 1) * w, (cy + box[3]/2).clamp(0, 1) * h), dim=-1).reshape(-1, 4)
    prob, labels = out[5:].softmax(0).max(0)
    scores = (out[0].sigmoid() * prob).flatten()
    labels = labels.flatten()
    mask = scores >= 0.3
    boxes, scores, labels = boxes[mask], scores[mask], labels[mask]
    keep = batched_nms(boxes, scores, labels, 0.5) if len(boxes) else []
    return [{"class_id": int(labels[i]), "score": float(scores[i]), "bbox": boxes[i].tolist()} for i in keep]

def classify(question):
    words = re.findall(r"\w+", question.lower())
    ids = ([vocab.get(word, 1) for word in words] + [0] * 32)[:32]
    text_model.set_tensor(text_input["index"], np.array([ids], dtype=text_input["dtype"]))
    text_model.invoke()
    return intents[int(text_model.get_tensor(text_output["index"])[0].argmax())]

with open(f"{root}/intent_test.csv", encoding="utf-8-sig") as f:
    questions = {row["question_id"]: row["question"] for row in csv.DictReader(f)}
with open(f"{root}/sample_submission.csv", encoding="utf-8-sig") as src, open("submission.csv", "w", newline="", encoding="utf-8") as dst:
    reader = csv.DictReader(src)
    writer = csv.DictWriter(dst, fieldnames=reader.fieldnames)
    writer.writeheader()
    for row in reader:
        if row["task_type"] == "DETECTION":
            row["prediction"] = json.dumps(detect(f"{root}/test/images/{row['sample_id']}.png"))
        elif row["task_type"] == "INTENT":
            row["prediction"] = classify(questions[row["sample_id"]])
        else:
            raise ValueError(row["task_type"])
        writer.writerow(row)
print("Saved submission.csv")